# Notebook 03: Rule-Based Baselines

**Purpose:** Design 3-rule OR-logic baselines for both datasets using thresholds derived from training data only. Evaluate on the test set.

**Outputs:** `results/baselines/`

In [1]:
import os
# Use the repository root as the working directory, whether this notebook is
# launched from the repo root or from the notebooks/ folder.
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

import pandas as pd
import numpy as np
import json
from sklearn.metrics import (f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, accuracy_score,
    matthews_corrcoef, balanced_accuracy_score, average_precision_score)

os.makedirs('results/baselines', exist_ok=True)
print('Ready.')

Ready.


## Part 1: UGRansome2024 Baseline

Rules follow the dissertation methodology: thresholds from training data only. Three rules using flow volume, protocol, and cluster membership.

In [2]:
ugr_train = pd.read_csv('data/processed/ugr_train.csv')
ugr_test  = pd.read_csv('data/processed/ugr_test.csv')

y_train_ugr = ugr_train['Prediction'].values
y_test_ugr  = ugr_test['Prediction'].values
X_train_ugr = ugr_train.drop(columns=['Prediction'])
X_test_ugr  = ugr_test.drop(columns=['Prediction'])

benign_mask = y_train_ugr == 0
attack_mask = y_train_ugr == 1

print(f'UGR train: {X_train_ugr.shape}, test: {X_test_ugr.shape}')

UGR train: (71887, 49), test: (17972, 49)


In [3]:
# Rule 1: Netflow_Bytes above 95th percentile of benign training flows
# High-volume flows are disproportionately observed in ransomware activity
netflow_thresh = np.percentile(X_train_ugr.loc[benign_mask, 'Netflow_Bytes'], 95)
print(f'Rule 1 threshold - Netflow_Bytes > {netflow_thresh:.2f}')

# Rule 2: Protocol is ICMP
# ICMP is more prevalent in attack flows (41.3%) than benign flows (24.2%)
# Measured on training data
icmp_attack_rate = X_train_ugr.loc[attack_mask, 'Protocol_ICMP'].mean()
icmp_benign_rate = X_train_ugr.loc[benign_mask, 'Protocol_ICMP'].mean()
print(f'Rule 2 - ICMP rate: attack={icmp_attack_rate:.3f}, benign={icmp_benign_rate:.3f}')

# Rule 3: Cluster 2 membership
# Cluster 2 is strongly associated with attack flows (64.3% vs 7.3% in benign)
c2_attack_rate = X_train_ugr.loc[attack_mask, 'Clusters_2'].mean()
c2_benign_rate = X_train_ugr.loc[benign_mask, 'Clusters_2'].mean()
print(f'Rule 3 - Cluster_2 rate: attack={c2_attack_rate:.3f}, benign={c2_benign_rate:.3f}')

Rule 1 threshold - Netflow_Bytes > 5976.00
Rule 2 - ICMP rate: attack=0.413, benign=0.242
Rule 3 - Cluster_2 rate: attack=0.643, benign=0.073


In [4]:
def apply_ugr_baseline(X):
    r1 = X['Netflow_Bytes'] > netflow_thresh
    r2 = X['Protocol_ICMP'] == 1
    r3 = X['Clusters_2'] == 1
    return (r1 | r2 | r3).astype(int).values

def compute_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    npv  = tn / (tn + fn) if (tn + fn) > 0 else 0
    fpr  = fp / (fp + tn) if (fp + tn) > 0 else 0
    return {
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'accuracy': accuracy_score(y_true, y_pred),
        'mcc': matthews_corrcoef(y_true, y_pred),
        'balanced_acc': balanced_accuracy_score(y_true, y_pred),
        'specificity': spec, 'npv': npv, 'fpr': fpr,
        'tp': int(tp), 'fp': int(fp), 'fn': int(fn), 'tn': int(tn)
    }

y_pred_ugr = apply_ugr_baseline(X_test_ugr)
ugr_base_metrics = compute_metrics(y_test_ugr, y_pred_ugr)
print('UGR Baseline metrics:')
for k, v in ugr_base_metrics.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

UGR Baseline metrics:
  f1_macro: 0.6274
  precision: 0.3828
  recall: 0.7056
  accuracy: 0.6735
  mcc: 0.3142
  balanced_acc: 0.6849
  specificity: 0.6641
  npv: 0.8843
  fpr: 0.3359
  tp: 2891
  fp: 4661
  fn: 1206
  tn: 9214


## Part 2: CICIoT2023 Baseline

In [5]:
cic_train = pd.read_csv('data/processed/cic_train.csv')
cic_test  = pd.read_csv('data/processed/cic_test.csv')

FEATURE_COLS_CIC = [c for c in cic_train.columns if c not in ['label', 'label_binary']]
y_train_cic = cic_train['label_binary'].values
y_test_cic  = cic_test['label_binary'].values
X_train_cic = cic_train[FEATURE_COLS_CIC]
X_test_cic  = cic_test[FEATURE_COLS_CIC]

benign_mask_c = y_train_cic == 0
attack_mask_c = y_train_cic == 1

print(f'CIC train: {X_train_cic.shape}, test: {X_test_cic.shape}')

CIC train: (159874, 46), test: (39995, 46)


In [6]:
# Rule 1: IAT above 95th percentile of benign flows
# High inter-arrival time is more common in attack bursts
iat_thresh = np.percentile(X_train_cic.loc[benign_mask_c, 'IAT'], 95)
print(f'Rule 1 threshold - IAT > {iat_thresh:.2f}')

# Rule 2: flow_duration below 5th percentile of benign
# Very short flows are characteristic of flood-type attacks
dur_thresh = np.percentile(X_train_cic.loc[benign_mask_c, 'flow_duration'], 5)
print(f'Rule 2 threshold - flow_duration < {dur_thresh:.6f}')

# Rule 3: Rate above 95th percentile of benign
# High packet rate is a strong indicator of flood attacks
rate_thresh = np.percentile(X_train_cic.loc[benign_mask_c, 'Rate'], 95)
print(f'Rule 3 threshold - Rate > {rate_thresh:.2f}')

Rule 1 threshold - IAT > 166525347.42
Rule 2 threshold - flow_duration < 0.702439
Rule 3 threshold - Rate > 838.47


In [7]:
def apply_cic_baseline(X):
    r1 = X['IAT'] > iat_thresh
    r2 = X['flow_duration'] < dur_thresh
    r3 = X['Rate'] > rate_thresh
    return (r1 | r2 | r3).astype(int).values

y_pred_cic = apply_cic_baseline(X_test_cic)
cic_base_metrics = compute_metrics(y_test_cic, y_pred_cic)
print('CIC Baseline metrics:')
for k, v in cic_base_metrics.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

CIC Baseline metrics:
  f1_macro: 0.5986
  precision: 0.9963
  recall: 0.8842
  accuracy: 0.8838
  mcc: 0.3347
  balanced_acc: 0.8745
  specificity: 0.8647
  npv: 0.1532
  fpr: 0.1353
  tp: 34529
  fp: 128
  fn: 4520
  tn: 818


## Part 3: Save results

In [8]:
metrics_df = pd.DataFrame([
    {'dataset': 'UGRansome2024', 'model': 'Baseline', **ugr_base_metrics},
    {'dataset': 'CICIoT2023',    'model': 'Baseline', **cic_base_metrics},
])
metrics_df.to_csv('results/baselines/baseline_metrics.csv', index=False)
print('Saved baseline_metrics.csv')
print(metrics_df[['dataset','f1_macro','precision','recall','mcc']].to_string())

Saved baseline_metrics.csv
         dataset  f1_macro  precision    recall       mcc
0  UGRansome2024  0.627431   0.382812  0.705638  0.314229
1     CICIoT2023  0.598641   0.996307  0.884248  0.334668


In [9]:
pd.DataFrame({'y_true': y_test_ugr, 'baseline_pred': y_pred_ugr}).to_csv(
    'results/baselines/ugr_baseline_predictions.csv', index=False)
pd.DataFrame({'y_true': y_test_cic, 'baseline_pred': y_pred_cic}).to_csv(
    'results/baselines/cic_baseline_predictions.csv', index=False)

rules = {
    'UGRansome2024': {
        'rule1': f'Netflow_Bytes > {netflow_thresh:.4f} (95th pct of benign training flows)',
        'rule2': 'Protocol_ICMP == 1 (ICMP rate 41.3% attack vs 24.2% benign)',
        'rule3': 'Clusters_2 == 1 (Cluster 2 rate 64.3% attack vs 7.3% benign)',
        'logic': 'OR'
    },
    'CICIoT2023': {
        'rule1': f'IAT > {iat_thresh:.4f} (95th pct of benign training flows)',
        'rule2': f'flow_duration < {dur_thresh:.8f} (5th pct of benign training flows)',
        'rule3': f'Rate > {rate_thresh:.4f} (95th pct of benign training flows)',
        'logic': 'OR'
    }
}
with open('results/baselines/rules.json', 'w') as f:
    json.dump(rules, f, indent=2)
print('Rules and predictions saved.')

Rules and predictions saved.


## Summary

**Purpose:** Establish simple rule-based detection baselines for both datasets.

**Method (UGRansome):** Three training-data rules with OR logic: high Netflow_Bytes (above the 95th percentile of benign flows), Protocol is ICMP (41.3% attack vs 24.2% benign prevalence in training), and Cluster 2 membership (64.3% attack vs 7.3% benign prevalence in training). Thresholds set from training data only.

**Method (CICIoT):** Three rules: IAT above 95th percentile of benign, flow_duration below 5th percentile of benign (short burst pattern), and Rate above 95th percentile of benign. OR logic. All thresholds from training data only.

**Key findings:** See printed F1 scores. ML models in Notebook 04 should substantially improve over these baselines.